# RTX4090 ASL custom-loss screening runner

Purpose: run the **ASL custom-loss screening round** only: 10 ASL-derived custom losses plus 2 baselines on YOLO models and the top-3 TIMM/Torch models.

Target environment:
- Linux amd64 local RTX4090
- Python 3.12.2
- Repo: `/home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26`
- Dataset: `processed-images/processed_images`

This notebook pulls the latest feature branch, installs only missing packages, redirects Kaggle dataset download calls to the local dataset, prepares manifests, runs screening with strict resume, collects ranked results, and audits YOLO class mapping outputs.

The notebook does **not** contain or store a GitHub token. Use an environment variable `GITHUB_TOKEN`, or paste it into the hidden prompt when asked. If you pasted a token into chat earlier, revoke it and create a new one.


In [2]:
import os
import sys
import json
import time
import shlex
import shutil
import getpass
import subprocess
from pathlib import Path
from datetime import datetime, timezone

WORKDIR = Path("/home/drnguyenvinh/notebooks").expanduser().resolve()
REPO_DIR = WORKDIR / "CVio_Shrimp_Disease_Classification_Capstone_SU26"
LOG_DIR = WORKDIR / "notebook_command_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/hntnhan1111-ayai/CVio_Shrimp_Disease_Classification_Capstone_SU26.git"
BRANCH_NAME = "feature/improving-lightweight-shrimp-disease-classification-coinfection-losses-randaugment"
DATASET_ROOT_INPUT = "processed-images/processed_images"
OUTPUT_DIR_ASL = WORKDIR / "shrimp_outputs_asl_custom_screening_rtx4090"

RUN_GIT_PULL = True
BACKUP_NON_GIT_REPO_DIR = True
FORCE_DELETE_NON_GIT_REPO_DIR = False

STRICT_PYTHON_3122 = False
RUN_INSTALL_MISSING_PACKAGES = True
INSTALL_TORCH_IF_MISSING = False
RUN_STATIC_CHECKS = True
RUN_PREPARE_DATASET = True
RUN_LIST_RUNS = True
RUN_VALIDATE_RESUME_BEFORE = True
RUN_YOLO_LOSS_SELF_CHECK = True
RUN_ASL_CUSTOM_SCREENING = True
RUN_VERIFY_ARTIFACTS = True
RUN_COLLECT_RESULTS = True
RUN_VALIDATE_RESUME_AFTER = True
RUN_SUMMARY_TABLES = True
RUN_BACKUP_OUTPUTS = True

SCREEN_BACKEND = "all"
SCREEN_MODEL = None
SCREEN_LOSS = None
SCREEN_RUN_ID = None
SCREEN_START = 0
SCREEN_LIMIT = None

EXPECTED_RUNS = 96

def utc_stamp() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

def normalize_branch_name(branch: str) -> str:
    branch = str(branch).strip()
    if branch.startswith("origin/"):
        return branch[len("origin/"):]
    return branch

BRANCH_NAME = normalize_branch_name(BRANCH_NAME)


def run_cmd(cmd, cwd=None, log_name="command", check=True, env=None, safe_cmd=None):
    cwd = Path(cwd or WORKDIR).expanduser().resolve()
    cwd.mkdir(parents=True, exist_ok=True)
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    log_path = LOG_DIR / f"{utc_stamp()}_{log_name}.log"
    shown = safe_cmd if safe_cmd is not None else cmd
    shown_text = " ".join(shlex.quote(str(x)) for x in shown)
    merged_env = os.environ.copy()
    merged_env["PYTHONUNBUFFERED"] = "1"
    merged_env["GIT_TERMINAL_PROMPT"] = "0"
    if env:
        merged_env.update({str(k): str(v) for k, v in env.items()})
    print("\n" + "=" * 110, flush=True)
    print("RUN:", shown_text, flush=True)
    print("CWD:", cwd, flush=True)
    print("LOG:", log_path, flush=True)
    print("=" * 110, flush=True)
    start = time.time()
    with open(log_path, "w", encoding="utf-8", errors="replace") as f:
        f.write("RUN: " + shown_text + "\n")
        f.write("CWD: " + str(cwd) + "\n\n")
        f.flush()
        proc = subprocess.Popen(
            [str(x) for x in cmd],
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=merged_env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            f.write(line)
            f.flush()
        rc = proc.wait()
    elapsed = time.time() - start
    print(f"\nRETURN_CODE={rc} | elapsed={elapsed:.1f}s | log={log_path}", flush=True)
    if check and rc != 0:
        raise RuntimeError(f"Command failed with return code {rc}: {shown_text}\nLog: {log_path}")
    return rc


def py_script(*args, cwd=None, log_name="python_script", check=True, env=None):
    return run_cmd([sys.executable, *map(str, args)], cwd=cwd or REPO_DIR, log_name=log_name, check=check, env=env)

print(json.dumps({
    "workdir": str(WORKDIR),
    "repo_dir": str(REPO_DIR),
    "dataset_input": DATASET_ROOT_INPUT,
    "output_dir_asl": str(OUTPUT_DIR_ASL),
    "branch": BRANCH_NAME,
    "expected_screening_runs": EXPECTED_RUNS,
}, indent=2))


{
  "workdir": "/home/drnguyenvinh/notebooks",
  "repo_dir": "/home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26",
  "dataset_input": "processed-images/processed_images",
  "output_dir_asl": "/home/drnguyenvinh/notebooks/shrimp_outputs_asl_custom_screening_rtx4090",
  "branch": "feature/improving-lightweight-shrimp-disease-classification-coinfection-losses-randaugment",
  "expected_screening_runs": 96
}


In [3]:
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", sys.platform)

if STRICT_PYTHON_3122 and sys.version_info[:3] != (3, 12, 2):
    raise RuntimeError(
        f"Wrong Python version: {sys.version.split()[0]}. "
        "Switch this notebook to the Python 3.12.2 RTX4090 kernel before running experiments."
    )


Python: 3.13.2 | packaged by Anaconda, Inc. | (main, Feb  6 2025, 18:56:02) [GCC 11.2.0]
Executable: /opt/miniconda3/bin/python
Platform: linux


In [4]:
def make_git_auth_env():
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if not token:
        token = getpass.getpass("GITHUB_TOKEN is not set. Paste token for private repo access, or press Enter for public clone: ").strip()
    env = {}
    if token:
        auth_dir = WORKDIR / ".git_auth_runtime"
        auth_dir.mkdir(parents=True, exist_ok=True)
        askpass = auth_dir / "git_askpass.py"
        askpass.write_text(
            "import os, sys\n"
            "prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\n"
            "if 'username' in prompt:\n"
            "    print(os.environ.get('GITHUB_USERNAME', 'x-access-token'))\n"
            "else:\n"
            "    print(os.environ.get('GITHUB_TOKEN', ''))\n",
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        env.update({
            "GIT_ASKPASS": str(askpass),
            "GITHUB_USERNAME": os.environ.get("GITHUB_USERNAME", "x-access-token"),
            "GITHUB_TOKEN": token,
            "GIT_TERMINAL_PROMPT": "0",
        })
        print("Git token loaded via hidden prompt/environment and will not be printed.")
    else:
        print("No token provided. Clone/pull works only if the repository is public.")
    return env


def backup_or_remove_non_git_dir(path: Path):
    path = Path(path).expanduser().resolve()
    if not path.exists() or (path / ".git").exists():
        return
    if not path.is_dir():
        raise RuntimeError(f"REPO_DIR exists but is not a directory: {path}")
    if FORCE_DELETE_NON_GIT_REPO_DIR:
        print(f"Deleting non-Git REPO_DIR because FORCE_DELETE_NON_GIT_REPO_DIR=True: {path}")
        shutil.rmtree(path)
        return
    if BACKUP_NON_GIT_REPO_DIR:
        backup_path = path.with_name(path.name + f"_non_git_backup_{utc_stamp()}")
        print("REPO_DIR exists but is not a Git checkout. Moving it to backup instead of deleting.")
        print("FROM:", path)
        print("TO:  ", backup_path)
        shutil.move(str(path), str(backup_path))
        return
    raise RuntimeError(
        f"REPO_DIR exists but is not a Git checkout: {path}. "
        "Set BACKUP_NON_GIT_REPO_DIR=True to move it aside."
    )

if RUN_GIT_PULL:
    WORKDIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORKDIR)
    git_env = make_git_auth_env()
    backup_or_remove_non_git_dir(REPO_DIR)
    if not REPO_DIR.exists():
        run_cmd(
            ["git", "clone", "--branch", BRANCH_NAME, "--single-branch", REPO_URL, str(REPO_DIR)],
            cwd=WORKDIR,
            log_name="git_clone",
            env=git_env,
        )
    else:
        run_cmd(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR, log_name="git_remote_plain", env=git_env)
        run_cmd(["git", "fetch", "origin", BRANCH_NAME], cwd=REPO_DIR, log_name="git_fetch", env=git_env)
        run_cmd(["git", "checkout", BRANCH_NAME], cwd=REPO_DIR, log_name="git_checkout", env=git_env)
        run_cmd(["git", "pull", "--ff-only", "origin", BRANCH_NAME], cwd=REPO_DIR, log_name="git_pull_ff_only", env=git_env)
    run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, log_name="git_branch")
    run_cmd(["git", "status", "--short"], cwd=REPO_DIR, log_name="git_status")
    run_cmd(["git", "log", "--oneline", "-5"], cwd=REPO_DIR, log_name="git_log")
else:
    print("Skipped Git pull. Set RUN_GIT_PULL=True to pull latest code.")

assert REPO_DIR.exists() and (REPO_DIR / ".git").exists(), f"Repo checkout is invalid: {REPO_DIR}"
os.chdir(REPO_DIR)
print("Current working directory:", Path.cwd())


Git token loaded via hidden prompt/environment and will not be printed.

RUN: git remote set-url origin https://github.com/hntnhan1111-ayai/CVio_Shrimp_Disease_Classification_Capstone_SU26.git
CWD: /home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26
LOG: /home/drnguyenvinh/notebooks/notebook_command_logs/20260602_064649_git_remote_plain.log

RETURN_CODE=0 | elapsed=0.0s | log=/home/drnguyenvinh/notebooks/notebook_command_logs/20260602_064649_git_remote_plain.log

RUN: git fetch origin feature/improving-lightweight-shrimp-disease-classification-coinfection-losses-randaugment
CWD: /home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26
LOG: /home/drnguyenvinh/notebooks/notebook_command_logs/20260602_064649_git_fetch.log
From https://github.com/hntnhan1111-ayai/CVio_Shrimp_Disease_Classification_Capstone_SU26
 * branch            feature/improving-lightweight-shrimp-disease-classification-coinfection-losses-randaugment -> FETCH_HEAD

RETURN_

In [5]:
import importlib.util

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "torchvision": "torchvision",
    "timm": "timm",
    "ultralytics": "ultralytics",
    "tqdm": "tqdm",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
    "yaml": "PyYAML",
    "cv2": "opencv-python",
}

missing = []
for module_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        missing.append((module_name, package_name))

print("Missing modules:", missing)

if RUN_INSTALL_MISSING_PACKAGES and missing:
    packages = []
    for module_name, package_name in missing:
        if module_name in {"torch", "torchvision"} and not INSTALL_TORCH_IF_MISSING:
            raise RuntimeError("torch/torchvision missing. Install the correct CUDA build manually or set INSTALL_TORCH_IF_MISSING=True.")
        packages.append(package_name)
    unique_packages = []
    for pkg in packages:
        if pkg not in unique_packages:
            unique_packages.append(pkg)
    run_cmd([sys.executable, "-m", "pip", "install", *unique_packages], cwd=REPO_DIR, log_name="pip_install_missing")
else:
    print("No missing packages or installation disabled.")

import torch
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This notebook is intended for the RTX4090 GPU environment.")


Missing modules: []
No missing packages or installation disabled.
torch: 2.10.0+cu128
cuda_available: True
cuda_device_count: 1
0 NVIDIA GeForce RTX 4090


In [6]:
def resolve_local_dataset_root(dataset_input: str | Path) -> Path:
    raw = Path(dataset_input).expanduser()
    if raw.is_absolute():
        base_candidates = [raw]
    else:
        base_candidates = [
            Path.cwd() / raw,
            REPO_DIR / raw,
            WORKDIR / raw,
            REPO_DIR.parent / raw,
        ]
    candidates = []
    for c in base_candidates:
        candidates.extend([
            c,
            c / "processed-images",
            c / "processed_images",
            c / "processed-images" / "processed_images",
            c / "processed_images" / "processed_images",
            c / "processed-images" / "processed-images",
        ])
    class_dirs = ["1. Healthy", "2. BG", "3. WSSV", "4. WSSV_BG"]
    attempts = []
    for c in candidates:
        r = c.resolve()
        ok = r.is_dir() and all((r / d).is_dir() for d in class_dirs)
        attempts.append({"candidate": str(r), "ok": ok})
        if ok:
            return r
    print(json.dumps({"dataset_resolution_attempts": attempts}, indent=2))
    raise FileNotFoundError("Could not find local dataset root containing expected class folders.")

DATASET_ROOT = resolve_local_dataset_root(DATASET_ROOT_INPUT)
OUTPUT_DIR_ASL.mkdir(parents=True, exist_ok=True)
print("DATASET_ROOT:", DATASET_ROOT)
print("OUTPUT_DIR_ASL:", OUTPUT_DIR_ASL)

runtime_overrides = REPO_DIR / "_local_runtime_overrides"
runtime_overrides.mkdir(parents=True, exist_ok=True)
(runtime_overrides / "kagglehub.py").write_text(
    "def dataset_download(dataset_id):\n"
    f"    return r'''{str(DATASET_ROOT)}'''\n",
    encoding="utf-8",
)

SCRIPT_ENV = os.environ.copy()
SCRIPT_ENV["PYTHONUNBUFFERED"] = "1"
SCRIPT_ENV["PYTHONPATH"] = str(runtime_overrides) + os.pathsep + str(REPO_DIR) + os.pathsep + SCRIPT_ENV.get("PYTHONPATH", "")
SCRIPT_ENV["SHRIMP_LOCAL_DATASET_ROOT"] = str(DATASET_ROOT)

print("Local kagglehub override:", runtime_overrides / "kagglehub.py")


DATASET_ROOT: /home/drnguyenvinh/notebooks/processed-images/processed_images
OUTPUT_DIR_ASL: /home/drnguyenvinh/notebooks/shrimp_outputs_asl_custom_screening_rtx4090
Local kagglehub override: /home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26/_local_runtime_overrides/kagglehub.py


In [7]:
if RUN_STATIC_CHECKS:
    py_script("-m", "compileall", "shrimp_scripts", "experiments", cwd=REPO_DIR, log_name="compileall", env=SCRIPT_ENV)
    required_files = [
        "shrimp_scripts/run_01_prepare_dataset.py",
        "experiments/asl_custom_loss_screening/run_asl_custom_screen.py",
        "experiments/asl_custom_loss_screening/collect_asl_custom_results.py",
        "shrimp_scripts/models_yolo.py",
        "shrimp_scripts/models_torch.py",
        "shrimp_scripts/losses.py",
    ]
    missing_files = [p for p in required_files if not (REPO_DIR / p).exists()]
    print("Missing required files:", missing_files)
    if missing_files:
        raise FileNotFoundError(missing_files)
    scan_targets = list((REPO_DIR / "shrimp_scripts").glob("*.py")) + list((REPO_DIR / "experiments").rglob("*.py"))
    violations = []
    for path in scan_targets:
        text = path.read_text(encoding="utf-8", errors="replace")
        bad_terms = [term for term in ["task=segment", "yolo26m-seg", "ExecuTorch", "NCNN", "TFLite"] if term in text]
        if bad_terms:
            violations.append({"file": str(path.relative_to(REPO_DIR)), "terms": bad_terms})
    print(json.dumps({"scope_violations": violations}, indent=2))
    if violations:
        raise RuntimeError("Unexpected segmentation/export terms found in scripts.")
else:
    print("Skipped static checks.")



RUN: /opt/miniconda3/bin/python -m compileall shrimp_scripts experiments
CWD: /home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26
LOG: /home/drnguyenvinh/notebooks/notebook_command_logs/20260602_064652_compileall.log
Listing 'shrimp_scripts'...
Compiling 'shrimp_scripts/attention.py'...
Compiling 'shrimp_scripts/config.py'...
Compiling 'shrimp_scripts/models_torch.py'...
Compiling 'shrimp_scripts/models_yolo.py'...
Compiling 'shrimp_scripts/report.py'...
Compiling 'shrimp_scripts/run_02_train_core_ablation.py'...
Compiling 'shrimp_scripts/run_03_train_yolo_family.py'...
Compiling 'shrimp_scripts/run_04_train_lightweight_models.py'...
Listing 'experiments'...
Listing 'experiments/asl_custom_loss_screening'...
Compiling 'experiments/asl_custom_loss_screening/screening_lib.py'...
Listing 'experiments/yolo_attention_screening'...
Compiling 'experiments/yolo_attention_screening/__init__.py'...
Compiling 'experiments/yolo_attention_screening/collect_yolo_attention_

In [8]:
if RUN_PREPARE_DATASET:
    py_script(
        "shrimp_scripts/run_01_prepare_dataset.py",
        "--output_dir", str(OUTPUT_DIR_ASL),
        "--resume",
        "--progress",
        cwd=REPO_DIR,
        log_name="prepare_dataset_for_asl_screening",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped dataset preparation.")



RUN: /opt/miniconda3/bin/python shrimp_scripts/run_01_prepare_dataset.py --output_dir /home/drnguyenvinh/notebooks/shrimp_outputs_asl_custom_screening_rtx4090 --resume --progress
CWD: /home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26
LOG: /home/drnguyenvinh/notebooks/notebook_command_logs/20260602_064652_prepare_dataset_for_asl_screening.log
[2026-06-02T06:46:53.048310+00:00] [INFO] Starting dataset preparation.
[2026-06-02T06:46:53.049114+00:00] [INFO] Downloading processed dataset with kagglehub.
Path to dataset files: /home/drnguyenvinh/notebooks/processed-images/processed_images
[2026-06-02T06:46:53.049241+00:00] [INFO] Dataset download completed.
[2026-06-02T06:46:53.049431+00:00] [INFO] Dataset root resolved.
[2026-06-02T06:46:53.059068+00:00] [INFO] Dataset class-count audit passed.
[2026-06-02T06:46:53.059373+00:00] [INFO] Building source/processed manifest with MD5 hashes.

manifest Healthy:  96%|█████████▋| 388/403 [00:00<00:00, 571.62it/s]
      

In [9]:
if RUN_LIST_RUNS:
    py_script(
        "experiments/asl_custom_loss_screening/run_asl_custom_screen.py",
        "--output_dir", str(OUTPUT_DIR_ASL),
        "--list_runs",
        cwd=REPO_DIR,
        log_name="list_asl_custom_screening_runs",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped ASL screening run listing.")



RUN: /opt/miniconda3/bin/python experiments/asl_custom_loss_screening/run_asl_custom_screen.py --output_dir /home/drnguyenvinh/notebooks/shrimp_outputs_asl_custom_screening_rtx4090 --list_runs
CWD: /home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26
LOG: /home/drnguyenvinh/notebooks/notebook_command_logs/20260602_064657_list_asl_custom_screening_runs.log
{
  "available_before_filter": 96,
  "selected_count": 96,
  "backend": "all",
  "model": null,
  "loss": null,
  "run_id": null,
  "start": 0,
  "limit": null,
  "runs": [
    {
      "run_id": "asl_custom_screening_timm_convnext_tiny_in22k_baseline_ce_seed42_repeat1",
      "screen_backend": "torch",
      "backend": "timm",
      "model": "convnext_tiny_in22k",
      "model_key": "convnext_tiny_in22k",
      "loss_key": "baseline_ce",
      "loss": "CE",
      "loss_role": "baseline",
      "condition_key": "baseline_ce",
      "randaugment": false,
      "experiment_key": "asl_custom_screening",
      "ex

In [10]:
if RUN_VALIDATE_RESUME_BEFORE:
    py_script(
        "experiments/asl_custom_loss_screening/run_asl_custom_screen.py",
        "--output_dir", str(OUTPUT_DIR_ASL),
        "--validate_resume",
        cwd=REPO_DIR,
        log_name="validate_resume_asl_before",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped pre-run resume validation.")



RUN: /opt/miniconda3/bin/python experiments/asl_custom_loss_screening/run_asl_custom_screen.py --output_dir /home/drnguyenvinh/notebooks/shrimp_outputs_asl_custom_screening_rtx4090 --validate_resume
CWD: /home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26
LOG: /home/drnguyenvinh/notebooks/notebook_command_logs/20260602_064700_validate_resume_asl_before.log
{
  "resume_validation": {
    "summary": {
      "planned_count": 96,
      "completed_with_valid_final_metrics": 22,
      "incomplete_missing_final_metrics": 50,
      "failed": 24,
      "skipped": 0,
      "will_fresh_rerun": 74,
      "skipped_completed_with_final_metrics": 22
    },
    "runs": [
      {
        "run_id": "asl_custom_screening_timm_convnext_tiny_in22k_baseline_ce_seed42_repeat1",
        "screen_backend": "torch",
        "backend": "timm",
        "model": "convnext_tiny_in22k",
        "loss_key": "baseline_ce",
        "validation_status": "incomplete_missing_final_metrics",
     

In [11]:
if RUN_YOLO_LOSS_SELF_CHECK:
    py_script(
        "experiments/asl_custom_loss_screening/run_asl_custom_screen.py",
        "--output_dir", str(OUTPUT_DIR_ASL),
        "--self_check_yolo_loss",
        cwd=REPO_DIR,
        log_name="yolo_custom_loss_self_check",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped YOLO custom-loss self-check.")



RUN: /opt/miniconda3/bin/python experiments/asl_custom_loss_screening/run_asl_custom_screen.py --output_dir /home/drnguyenvinh/notebooks/shrimp_outputs_asl_custom_screening_rtx4090 --self_check_yolo_loss
CWD: /home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26
LOG: /home/drnguyenvinh/notebooks/notebook_command_logs/20260602_064701_yolo_custom_loss_self_check.log
{
  "yolo_custom_loss_self_check": {
    "status": "passed",
    "loss_keys": [
      "baseline_ce",
      "asl_single_label",
      "class_weighted_asl",
      "coinfection_weighted_asl",
      "boundary_weighted_asl",
      "confusion_aware_negative_asl",
      "soft_target_coinfection_asl",
      "attribute_projection_asl",
      "adaptive_gamma_asl",
      "asl_ldam_margin",
      "dangerous_confidence_penalty_asl",
      "coinfection_logit_adjusted_asl"
    ],
    "checks": {
      "torch_available": true,
      "custom_classes_importable": true,
      "all_losses_passed": true
    },
    "loss_r

In [ ]:
def build_screening_command():
    cmd = [
        "experiments/asl_custom_loss_screening/run_asl_custom_screen.py",
        "--output_dir", str(OUTPUT_DIR_ASL),
        "--backend", SCREEN_BACKEND,
        "--resume",
        "--progress",
    ]
    if SCREEN_MODEL:
        cmd += ["--model", SCREEN_MODEL]
    if SCREEN_LOSS:
        cmd += ["--loss", SCREEN_LOSS]
    if SCREEN_RUN_ID:
        cmd += ["--run_id", SCREEN_RUN_ID]
    if SCREEN_START:
        cmd += ["--start", str(SCREEN_START)]
    if SCREEN_LIMIT is not None:
        cmd += ["--limit", str(SCREEN_LIMIT)]
    return cmd

if RUN_ASL_CUSTOM_SCREENING:
    cmd = build_screening_command()
    print("Launching ASL custom-loss screening command:")
    print(" ".join(map(str, [sys.executable, *cmd])))
    py_script(*cmd, cwd=REPO_DIR, log_name="run_asl_custom_loss_screening", env=SCRIPT_ENV)
else:
    print("Skipped ASL custom-loss screening.")


Launching ASL custom-loss screening command:
/opt/miniconda3/bin/python experiments/asl_custom_loss_screening/run_asl_custom_screen.py --output_dir /home/drnguyenvinh/notebooks/shrimp_outputs_asl_custom_screening_rtx4090 --backend all --resume --progress

RUN: /opt/miniconda3/bin/python experiments/asl_custom_loss_screening/run_asl_custom_screen.py --output_dir /home/drnguyenvinh/notebooks/shrimp_outputs_asl_custom_screening_rtx4090 --backend all --resume --progress
CWD: /home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26
LOG: /home/drnguyenvinh/notebooks/notebook_command_logs/20260602_064703_run_asl_custom_loss_screening.log
[2026-06-02T06:47:04.952067+00:00] [INFO] Starting ASL custom-loss screening.
[2026-06-02T06:47:04.961415+00:00] [INFO] [asl_custom_screening_timm_convnext_tiny_in22k_baseline_ce_seed42_repeat1] Starting ASL screening run.
[2026-06-02T06:47:04.961633+00:00] [INFO] [asl_custom_screening_timm_convnext_tiny_in22k_baseline_ce_seed42_repeat1] 

In [ ]:
if RUN_VERIFY_ARTIFACTS:
    py_script(
        "experiments/asl_custom_loss_screening/run_asl_custom_screen.py",
        "--output_dir", str(OUTPUT_DIR_ASL),
        "--verify_artifacts",
        "--skip_checkpoint_load",
        cwd=REPO_DIR,
        log_name="verify_asl_screening_artifacts",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped artifact verification.")

if RUN_COLLECT_RESULTS:
    py_script(
        "experiments/asl_custom_loss_screening/collect_asl_custom_results.py",
        "--output_dir", str(OUTPUT_DIR_ASL),
        cwd=REPO_DIR,
        log_name="collect_asl_custom_results",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped result collection.")


In [ ]:
if RUN_VALIDATE_RESUME_AFTER:
    py_script(
        "experiments/asl_custom_loss_screening/run_asl_custom_screen.py",
        "--output_dir", str(OUTPUT_DIR_ASL),
        "--validate_resume",
        cwd=REPO_DIR,
        log_name="validate_resume_asl_after",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped post-run resume validation.")


In [ ]:
import pandas as pd

def read_json_safe(path):
    path = Path(path)
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        return {"read_error": repr(exc)}

if RUN_SUMMARY_TABLES:
    expected_files = [
        "asl_custom_screening_summary.csv",
        "asl_custom_screening_ranked.csv",
        "asl_custom_screening_failures.csv",
        "asl_custom_screening_artifact_verification.csv",
        "asl_custom_screening_resume_validation.csv",
    ]
    for name in expected_files:
        p = OUTPUT_DIR_ASL / name
        print("\n" + "=" * 100)
        print(name, "EXISTS=", p.exists(), "SIZE=", p.stat().st_size if p.exists() else None)
        if p.exists() and p.suffix == ".csv":
            df = pd.read_csv(p)
            print("shape:", df.shape)
            display(df.head(20))

    ranked_path = OUTPUT_DIR_ASL / "asl_custom_screening_ranked.csv"
    if ranked_path.exists():
        ranked = pd.read_csv(ranked_path)
        metric_cols = [c for c in ["model", "loss_key", "loss_role", "test_macro_f1", "cohen_kappa", "beats_ce_and_asl_same_model", "screening_candidate"] if c in ranked.columns]
        sort_cols = [c for c in ["test_macro_f1", "cohen_kappa"] if c in ranked.columns]
        if metric_cols and sort_cols:
            print("\nTop ranked rows by Test Macro-F1:")
            display(ranked[metric_cols].sort_values(sort_cols, ascending=False).head(30))

    audit_paths = sorted((OUTPUT_DIR_ASL / "runs").glob("*/class_order_audit.json"))
    print("\nYOLO class_order_audit.json count:", len(audit_paths))
    audit_rows = []
    for p in audit_paths:
        d = read_json_safe(p)
        swap = d.get("swap_diagnostic") or {}
        audit_rows.append({
            "run_id": p.parent.name,
            "audit_passed": d.get("audit_passed"),
            "class_order_match": d.get("class_order_match"),
            "macro_f1_delta_after_swap": swap.get("macro_f1_delta_after_swap"),
            "failed_due_to_swap_diagnostic": swap.get("failed_due_to_swap_diagnostic"),
        })
    if audit_rows:
        audit_df = pd.DataFrame(audit_rows)
        display(audit_df.sort_values(["audit_passed", "macro_f1_delta_after_swap"], ascending=[True, False]).head(50))
        bad = audit_df[(audit_df["audit_passed"] != True) | (audit_df["failed_due_to_swap_diagnostic"] == True)]
        if len(bad):
            raise RuntimeError(f"YOLO class-order audit found {len(bad)} problematic runs. Inspect class_order_audit.json before paper claims.")
else:
    print("Skipped summary tables.")


In [ ]:
if RUN_BACKUP_OUTPUTS:
    import zipfile
    backup_path = WORKDIR / f"asl_custom_screening_rtx4090_no_weights_backup_{utc_stamp()}.zip"
    excluded_suffixes = {".pt", ".pth", ".onnx", ".engine", ".tflite"}
    with zipfile.ZipFile(backup_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root in [OUTPUT_DIR_ASL, LOG_DIR]:
            root = Path(root)
            if not root.exists():
                continue
            for path in root.rglob("*"):
                if not path.is_file():
                    continue
                if path.suffix.lower() in excluded_suffixes:
                    continue
                try:
                    zf.write(path, arcname=str(path.relative_to(WORKDIR)))
                except ValueError:
                    zf.write(path, arcname=path.name)
    print("Backup created:", backup_path)
else:
    print("Skipped backup.")
